# Final Evaluation Summary — Cultural Atlas Project

This notebook is the **final scientific summary of the frozen evaluation stage**.

Its purpose is not to rerun modelling. Instead, it:

- loads the evaluation outputs already produced by the project;
- checks that the expected result files are present;
- presents the main quantitative and qualitative evidence in one place;
- explains **what each metric evaluates**;
- records **what can and cannot be concluded** from each result;
- provides a compact reference for writing the final **Results, Discussion, Limitations and Conclusion** sections of the report.

> **Recommended use:** run this notebook from the project root, then keep it open while writing the report.

The notebook separates three questions:

1. **Semantic quality** — are nearby items meaningfully related in the representation?
2. **Projection fidelity** — does the 2D atlas preserve local relationships from the higher-dimensional representation?
3. **Cross-domain behaviour** — do the General Semantic and Feel approaches produce the kinds of cross-domain relationships they were designed to produce?

## 0. Evaluation philosophy

This project is not a conventional classification problem with a single ground-truth label and one final accuracy score.

An atlas can be useful while still being imperfect in several different ways. Evaluation therefore needs to be **multi-dimensional**:

| Evaluation question | Typical evidence |
|---|---|
| Are semantically similar items located near one another? | neighbourhood/category/tag coherence, lift |
| Does the 2D projection preserve the original representation? | trustworthiness |
| Are clusters interpretable? | cluster composition, representative items, qualitative inspection |
| Does a cross-domain method actually mix domains? | cross-domain neighbour share |
| Do Method A and Method B behave differently in the intended way? | matched-population comparison |
| Are Feel dimensions meaningful and non-redundant? | correlation, extremes, domain summaries |
| Are discovered analogies interpretable to a human? | qualitative neighbour examples |

**No individual metric validates the entire atlas.** The final argument is built from converging quantitative and qualitative evidence.

In [2]:
from pathlib import Path
from IPython.display import display, Markdown
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 120)

# ---------------------------------------------------------
# PROJECT PATHS
# ---------------------------------------------------------

PROJECT_ROOT = Path("../")

# Change this single path if your evaluation folder lives elsewhere.
EVALUATION_ROOT = PROJECT_ROOT / "data" / "processed" / "evaluation"

print("Project root:", PROJECT_ROOT.resolve())
print("Evaluation root:", EVALUATION_ROOT.resolve())

Project root: C:\Users\antoi\Documents\All_Files\Projects\University\FYP\Semantic_Atlas_Universe
Evaluation root: C:\Users\antoi\Documents\All_Files\Projects\University\FYP\Semantic_Atlas_Universe\data\processed\evaluation


### Path note

The default evaluation path is:

```text
data/processed/evaluation/
```

If your actual folder has another name, change only `EVALUATION_ROOT` in the previous cell. Everything else uses paths relative to it.

In [ ]:
FILES = {
    # Root-level fusion-weight experiments
    "movie_fusion_weights":
        EVALUATION_ROOT / "movie_fusion_weight_evaluation.csv",
    "music_fusion_weights":
        EVALUATION_ROOT / "music_fusion_weight_evaluation.csv",
    "restaurant_fusion_weights":
        EVALUATION_ROOT / "restaurant_fusion_weight_evaluation.csv",

    # Mono-domain evaluation
    "mono_atlas_summary":
        EVALUATION_ROOT / "mono_domain" / "mono_atlas_summary.csv",
    "mono_projection_fidelity":
        EVALUATION_ROOT / "mono_domain" / "mono_projection_fidelity.csv",
    "movies_semantic_coherence":
        EVALUATION_ROOT / "mono_domain" / "movies_semantic_coherence.csv",
    "music_semantic_coherence":
        EVALUATION_ROOT / "mono_domain" / "music_semantic_coherence.csv",
    "restaurants_semantic_coherence":
        EVALUATION_ROOT / "mono_domain" / "restaurants_semantic_coherence.csv",

    # Atlas comparison
    "atlas_metrics_summary":
        EVALUATION_ROOT / "atlas_comparison" / "atlas_metrics_summary.csv",
    "cross_domain_comparison":
        EVALUATION_ROOT / "atlas_comparison" / "cross_domain_comparison.csv",

    "composition_movies_music":
        EVALUATION_ROOT / "atlas_comparison" / "cluster_composition" / "movies_music.csv",
    "composition_movies_music_feel":
        EVALUATION_ROOT / "atlas_comparison" / "cluster_composition" / "movies_music_feel.csv",
    "composition_movies_music_restaurants":
        EVALUATION_ROOT / "atlas_comparison" / "cluster_composition" / "movies_music_restaurants.csv",
    "composition_movies_music_restaurants_feel":
        EVALUATION_ROOT / "atlas_comparison" / "cluster_composition" / "movies_music_restaurants_feel.csv",

    # Matched cross-domain evaluation
    "matched_atlas_metrics":
        EVALUATION_ROOT / "cross_domain_matched" / "matched_atlas_metrics.csv",
    "matched_method_comparison":
        EVALUATION_ROOT / "cross_domain_matched" / "matched_method_comparison.csv",
    "matched_population_summary":
        EVALUATION_ROOT / "cross_domain_matched" / "matched_population_summary.csv",

    "matched_composition_mm_a":
        EVALUATION_ROOT / "cross_domain_matched" / "cluster_composition" / "movies_music_method_a.csv",
    "matched_composition_mm_b":
        EVALUATION_ROOT / "cross_domain_matched" / "cluster_composition" / "movies_music_method_b.csv",
    "matched_composition_mmr_a":
        EVALUATION_ROOT / "cross_domain_matched" / "cluster_composition" / "movies_music_restaurants_method_a.csv",
    "matched_composition_mmr_b":
        EVALUATION_ROOT / "cross_domain_matched" / "cluster_composition" / "movies_music_restaurants_method_b.csv",

    # Cross-domain qualitative evaluation
    "cross_domain_neighbor_examples":
        EVALUATION_ROOT / "cross_domain_qualitative" / "cross_domain_neighbor_examples.csv",
    "cross_domain_query_summary":
        EVALUATION_ROOT / "cross_domain_qualitative" / "cross_domain_query_summary.csv",

    # Feel-space evaluation
    "feel_correlation_matrix":
        EVALUATION_ROOT / "feel_space" / "correlation_matrix.csv",
    "feel_dimension_extremes":
        EVALUATION_ROOT / "feel_space" / "dimension_extremes.csv",
    "feel_domain_dimension_summary":
        EVALUATION_ROOT / "feel_space" / "domain_dimension_summary.csv",
    "feel_domain_effects":
        EVALUATION_ROOT / "feel_space" / "domain_effects.csv",
    "feel_global_dimension_summary":
        EVALUATION_ROOT / "feel_space" / "global_dimension_summary.csv",
    "feel_high_correlations":
        EVALUATION_ROOT / "feel_space" / "high_correlations.csv",
    "feel_modality_summary":
        EVALUATION_ROOT / "feel_space" / "modality_summary.csv",

    # Feel-space qualitative outputs
    "feel_dimension_extremes_named":
        EVALUATION_ROOT / "feel_space" / "qualitative" / "dimension_extremes_named.csv",
    "feel_known_item_profiles":
        EVALUATION_ROOT / "feel_space" / "qualitative" / "known_item_profiles.csv",
    "feel_name_resolution_summary":
        EVALUATION_ROOT / "feel_space" / "qualitative" / "name_resolution_summary.csv",
}

In [ ]:
tables = {}
inventory_rows = []

for key, path in FILES.items():
    exists = path.exists()

    inventory_rows.append({
        "key": key,
        "exists": exists,
        "path": str(path),
    })

    if exists:
        try:
            tables[key] = pd.read_csv(path)
        except Exception as exc:
            print(f"Could not load {key}: {exc}")

inventory = pd.DataFrame(inventory_rows)
display(inventory)

print()
print(f"Loaded {len(tables)} / {len(FILES)} expected CSV files.")

### If files are marked `False`

Do not edit every path individually. Inspect where the evaluation folder actually lives and update only `EVALUATION_ROOT`, then rerun the loading cells.

In [ ]:
def show_table(key, rows=None):
    if key not in tables:
        print(f"[Missing] {key}")
        return

    df = tables[key]
    print(f"{key}: {len(df):,} rows × {len(df.columns)} columns")

    if rows is None:
        display(df)
    else:
        display(df.head(rows))


def numeric_summary(key):
    if key not in tables:
        print(f"[Missing] {key}")
        return

    numeric = tables[key].select_dtypes(include=np.number)

    if numeric.empty:
        print(f"No numeric columns in {key}.")
        return

    display(numeric.describe().T)


def plot_numeric_columns(key, title=None, max_columns=6):
    if key not in tables:
        print(f"[Missing] {key}")
        return

    df = tables[key]
    numeric_cols = list(df.select_dtypes(include=np.number).columns)[:max_columns]

    if not numeric_cols:
        print(f"No numeric columns available in {key}.")
        return

    ax = df[numeric_cols].plot(kind="bar", figsize=(10, 5))
    ax.set_title(title or key)
    ax.set_xlabel("Row")
    ax.set_ylabel("Value")
    plt.tight_layout()
    plt.show()

# 1. Mono-domain atlases

The mono-domain evaluation asks:

> **Within Movies, Music and Restaurants, does local atlas proximity correspond to meaningful domain-specific similarity?**

## 1.1 Semantic neighbourhood coherence / lift

A neighbourhood-coherence metric compares the rate of shared semantic characteristics among nearby atlas items with a baseline expectation.

Conceptually:

```text
lift > 1
```

means nearby items share the evaluated characteristic **more often than expected from the baseline**.

Examples used in this project include:

- Movie genre / macro-genre overlap;
- Music tag overlap;
- Restaurant category overlap.

A high lift supports the claim that the learned representation contains meaningful local semantic structure.

It does **not** prove that every neighbour relationship is subjectively correct.

In [ ]:
show_table("mono_atlas_summary")

In [ ]:
show_table("movies_semantic_coherence")

In [ ]:
show_table("music_semantic_coherence")

In [ ]:
show_table("restaurants_semantic_coherence")

### Frozen interpretation from the final project run

The retained final results were approximately:

| Domain | Main semantic-coherence result |
|---|---:|
| Movies | **1.82×** genre lift |
| Movies | **1.95×** macro-genre lift |
| Music | **7.03×** tag lift |
| Restaurants | **2.75×** category lift |

### Interpretation

- **Movies:** local neighbourhoods are meaningfully genre-enriched, although genres are broad and multi-label.
- **Music:** tag coherence is particularly strong, indicating highly structured local semantic neighbourhoods.
- **Restaurants:** category lift provides clear evidence that nearby venues tend to share cuisine / venue characteristics more often than the baseline.

### Defensible conclusion

> All three mono-domain representations exhibit measurable semantic organisation. The strength of that organisation varies by domain and by the available semantic labels, with Music showing particularly strong tag-based local coherence.

### Limitation

These tests validate only the semantic concept represented by the available labels. Genre overlap, for example, cannot capture every legitimate similarity between two films.

## 1.2 Projection fidelity — Trustworthiness

The semantic representations exist in a higher-dimensional feature space, but users explore them through a **2D UMAP projection**.

Trustworthiness asks:

> **When two items appear close in 2D, were they also genuine neighbours in the original high-dimensional representation?**

The score lies between 0 and 1:

- `1.0` = perfect local neighbourhood preservation;
- lower values = more false neighbours introduced by the 2D projection.

This matters because the frontend displays **visual 2D proximity**.

In [ ]:
show_table("mono_projection_fidelity")

In [ ]:
plot_numeric_columns(
    "mono_projection_fidelity",
    title="Mono-domain projection fidelity"
)

### Frozen interpretation from the final run

Approximate trustworthiness values:

| Domain | Trustworthiness |
|---|---:|
| Movies | **0.6976** |
| Music | **0.8216** |
| Restaurants | **0.9566** |

### Interpretation

- **Restaurants:** excellent local preservation.
- **Music:** strong local preservation, although some distortion remains.
- **Movies:** noticeably weaker preservation; fine-grained visual distance should therefore be interpreted more cautiously.

### Defensible conclusion

> The UMAP projection preserves local structure to different degrees across domains. Restaurant and Music maps retain strong local fidelity, while the Movie projection involves greater compression/distortion.

### Limitation

Trustworthiness evaluates preservation of the **model's representation**, not whether that representation corresponds perfectly to human cultural judgement.

# 2. Fusion-weight experiments

The mono-domain pipelines combine two semantic information sources:

- base semantic information such as tags/categories;
- review-derived semantic embeddings.

The fusion-weight experiments test whether changing the relative contribution of these components materially changes evaluation quality.

These are mainly useful as **model-selection evidence**, not as headline final results.

In [ ]:
for key in [
    "movie_fusion_weights",
    "music_fusion_weights",
    "restaurant_fusion_weights",
]:
    show_table(key)
    print()

In [ ]:
for key, title in [
    ("movie_fusion_weights", "Movie fusion-weight evaluation"),
    ("music_fusion_weights", "Music fusion-weight evaluation"),
    ("restaurant_fusion_weights", "Restaurant fusion-weight evaluation"),
]:
    plot_numeric_columns(key, title=title)

### How to discuss fusion weights

Unless the experiments show a dramatic optimum, use them conservatively:

> The fusion experiment was used to verify that the selected weighting offered a reasonable balance between structured semantic metadata and review-derived semantic information.

Avoid implying that a small difference between neighbouring weights proves a universally optimal parameter.

# 3. Global atlas comparison

These files compare the final atlas variants at a higher level.

They are useful for summarising:

- population size;
- clustering behaviour;
- domain mixing;
- broad structural differences between General Semantic and Feel atlases.

In [ ]:
show_table("atlas_metrics_summary")

In [ ]:
show_table("cross_domain_comparison")

## Cluster composition

Cluster-composition tables are descriptive rather than a single quality score.

They answer questions such as:

- Is a region almost entirely one domain?
- Are clusters genuinely cross-domain?
- Does one cluster contain most of the population?
- How fragmented or concentrated is the clustering?

This is especially important for the Feel atlases: a large cluster does **not** automatically mean the representation failed. It can indicate a broad continuous manifold that the clustering algorithm does not naturally partition into many discrete groups.

In [ ]:
for key in [
    "composition_movies_music",
    "composition_movies_music_feel",
    "composition_movies_music_restaurants",
    "composition_movies_music_restaurants_feel",
]:
    show_table(key, rows=20)
    print()

### Important frozen observation

The Feel atlases showed strong giant-cluster behaviour:

- Movies + Music Feel: largest cluster roughly **98%**.
- Movies + Music + Restaurants Feel: largest cluster roughly **83%**.

This should be reported as a limitation of **discrete cluster interpretation**, not automatically as failure of the continuous atlas.

The frontend therefore gives greater interpretive value to:

- local nearest neighbours;
- continuous Feel profiles;
- individual dimension values;
- map position.

LLM-generated cluster names remain a **post-hoc interface aid**.

# 4. Matched cross-domain evaluation — core Method A vs Method B test

This is arguably the most important direct comparison in the project.

## Why matched populations are necessary

Method A and Method B can contain different numbers of semantically defined entities.

A naive comparison could therefore confuse a **method difference** with a **population difference**.

The matched evaluation restricts both approaches to the **same entities**, making the comparison substantially fairer.

The key question is:

> **When evaluated on the same items, does the Feel representation produce more cross-domain local neighbourhoods than the General Semantic representation?**

In [ ]:
show_table("matched_population_summary")

In [ ]:
show_table("matched_atlas_metrics")

In [ ]:
show_table("matched_method_comparison")

### Frozen matched-population result

#### Movies + Music

```text
General Semantic (Method A):  ~0.0220 cross-domain neighbour share
Feel (Method B):              ~0.2061 cross-domain neighbour share
```

Approximately a **9.4× increase**.

#### Movies + Music + Restaurants

```text
General Semantic (Method A):  ~0.0178
Feel (Method B):              ~0.1575
```

Approximately an **8.8× increase**.

### What this means

> **Changing the shared representation changes the nature of cross-domain proximity in a large and measurable way.**

Method A tends to preserve domain-specific/literal semantic structure.

Method B substantially increases the frequency with which local neighbours come from another domain.

### What this does NOT prove

It does **not** prove that Feel similarity is objectively better.

It supports the narrower claim that the experiential representation better achieves the project's design objective of creating cross-domain neighbourhoods based on a shared Feel space.

In [ ]:
plot_numeric_columns(
    "matched_method_comparison",
    title="Matched Method A vs Method B comparison"
)

## Matched cluster composition

These files show whether the structural differences visible in aggregate metrics also appear at cluster level.

In [ ]:
for key in [
    "matched_composition_mm_a",
    "matched_composition_mm_b",
    "matched_composition_mmr_a",
    "matched_composition_mmr_b",
]:
    show_table(key, rows=20)
    print()

# 5. Cross-domain qualitative evaluation

Cross-domain neighbour share tells us **how often** domains mix.

It does not tell us whether the resulting relationships make sense.

The qualitative evaluation therefore inspects concrete query items and their neighbours:

> **When cross-domain neighbours appear, are at least some relationships interpretable in terms of topic, mood, atmosphere, intensity, warmth, nostalgia, etc.?**

In [ ]:
show_table("cross_domain_query_summary")

In [ ]:
show_table("cross_domain_neighbor_examples", rows=100)

### Interpretation

A useful qualitative finding is not:

> “Every recommendation is correct.”

A stronger formulation is:

> Examples demonstrate that the Feel representation can produce interpretable cross-domain analogies that would be difficult to obtain from literal semantic matching alone.

Counterexamples should also be acknowledged where useful.

Imperfect neighbours are expected because:

- the 13 experiential dimensions are an engineered abstraction;
- text embeddings only approximate perceived experience;
- UMAP introduces projection distortion;
- subjective cultural similarity has no single universal ground truth.

# 6. Feel-space representation evaluation

Before trusting the Feel atlas, the 13-dimensional experiential representation itself needs inspection.

The outputs address:

1. Are dimensions numerically well-behaved?
2. Are some dimensions almost duplicates?
3. Do domains occupy systematically different parts of the Feel space?
4. Do high/low extremes correspond to recognisable examples?
5. Are known-item profiles intuitively plausible?

## 6.1 Global dimension summary

In [ ]:
show_table("feel_global_dimension_summary")
numeric_summary("feel_global_dimension_summary")

## 6.2 Correlation between Feel dimensions

Correlation is used as a **redundancy diagnostic**.

Very high absolute correlation suggests two dimensions may capture overlapping information.

Moderate correlation is not inherently problematic: experiential concepts naturally overlap.

The 13 dimensions should be described as **engineered semantic axes**, not statistically independent psychological factors.

In [ ]:
show_table("feel_high_correlations")

In [ ]:
if "feel_correlation_matrix" in tables:
    corr_df = tables["feel_correlation_matrix"].copy()

    if len(corr_df.columns) > 1 and not pd.api.types.is_numeric_dtype(corr_df.iloc[:, 0]):
        corr_df = corr_df.set_index(corr_df.columns[0])

    numeric_corr = corr_df.apply(pd.to_numeric, errors="coerce")

    fig, ax = plt.subplots(figsize=(9, 7))
    image = ax.imshow(numeric_corr.values, aspect="auto", vmin=-1, vmax=1)
    ax.set_title("Feel dimension correlation matrix")
    ax.set_xticks(range(len(numeric_corr.columns)))
    ax.set_xticklabels(numeric_corr.columns, rotation=90)
    ax.set_yticks(range(len(numeric_corr.index)))
    ax.set_yticklabels(numeric_corr.index)
    fig.colorbar(image, ax=ax)
    plt.tight_layout()
    plt.show()
else:
    print("[Missing] feel_correlation_matrix")

## 6.3 Domain summaries and domain effects

These outputs investigate whether Movies, Music and Restaurants receive systematically different Feel profiles.

A domain effect does not invalidate cross-domain comparison, but it means the report should avoid claiming complete domain invariance.

> Shared dimensionality should not be confused with identical domain distributions.

In [ ]:
show_table("feel_domain_dimension_summary")

In [ ]:
show_table("feel_domain_effects")

In [ ]:
show_table("feel_modality_summary")

The matched cross-domain neighbour experiment remains especially important because it tests the **practical consequence** of the representation—actual local domain mixing—rather than assuming comparability from the dimensions alone.

## 6.4 Dimension extremes

Extreme examples provide an intuitive sanity check.

If items at the positive/negative ends of dimensions such as valence, activation, tension, warmth, nostalgia, wonder and tenderness broadly resemble those concepts, this supports **construct interpretability**.

This remains qualitative evidence, not formal psychological validation.

In [ ]:
show_table("feel_dimension_extremes")

In [ ]:
show_table("feel_dimension_extremes_named")

## 6.5 Known-item profiles

In [ ]:
show_table("feel_known_item_profiles")

In [ ]:
show_table("feel_name_resolution_summary")

Known-item profiles are useful explanatory examples:

> “Does the generated profile broadly align with how the item can reasonably be characterised?”

They are not formal accuracy tests because no authoritative 13-dimensional ground-truth labels exist.

# 7. Overall results synthesis

## Strongest supported conclusions

### A. Mono-domain atlases contain meaningful local semantic structure

Neighbourhood lift is above baseline for all three domains:

- Movies: meaningful genre enrichment;
- Music: particularly strong tag enrichment;
- Restaurants: clear category enrichment.

### B. 2D projection quality varies by domain

Trustworthiness indicates:

- Restaurant 2D structure is highly faithful;
- Music is strongly preserved;
- Movies have substantially more projection distortion.

### C. General Semantic and Feel methods solve different problems

Method A:

> **literal / topical semantic organisation**

Method B:

> **shared experiential organisation**

Neither should be declared universally superior.

### D. Feel substantially increases cross-domain local mixing

Matched comparisons show approximately:

- **9.4×** greater cross-domain neighbour share for Movies + Music;
- **8.8×** greater cross-domain neighbour share for Movies + Music + Restaurants.

### E. Cross-domain Feel similarity is interpretable but not universal

Qualitative examples support interpretable experiential analogies across domains, while subjective similarity remains imperfect and context-dependent.

### F. Feel clustering is a weak summary of a continuous space

Giant-cluster behaviour means discrete cluster labels should not be the primary explanatory mechanism for Feel.

Local neighbours and continuous profiles are stronger interpretive tools.

# 8. What the project does **not** establish

The project does **not** demonstrate that:

1. 2D Euclidean distance is a perfect measure of cultural similarity.
2. UMAP preserves all global relationships from the original embedding.
3. The 13 Feel dimensions are psychologically independent or experimentally validated.
4. Cross-domain Feel neighbours represent a universal human notion of similarity.
5. Cluster membership is a ground-truth cultural taxonomy.
6. LLM-generated cluster names validate the clusters scientifically.
7. One representation is globally “better” than the other.

Instead, the project demonstrates that different engineered representations generate **measurably different, interpretable structures** that can be explored through an interactive research tool.

# 9. Final report-ready interpretation

> The project demonstrates that large heterogeneous cultural datasets can be transformed into interactive semantic atlases whose local organisation is quantitatively measurable and qualitatively interpretable. Mono-domain evaluations show meaningful semantic neighbourhood enrichment, while trustworthiness analysis reveals domain-dependent distortion introduced by 2D projection. In cross-domain experiments, a shared literal semantic representation largely preserves domain separation, whereas the engineered experiential representation produces substantially higher cross-domain neighbour mixing on matched populations. This suggests that cross-domain cultural proximity is strongly dependent on the representation used: literal semantics and shared experiential character expose different forms of similarity. The resulting atlases should therefore be treated as exploratory representations rather than objective cultural taxonomies, with local neighbourhoods and continuous profiles providing stronger evidence than discrete cluster assignments alone.

# 10. Quick results checklist for report writing

Before finalising the Results chapter, verify each item directly against the loaded CSVs.

### Mono-domain

- [ ] Movie semantic lift quoted correctly
- [ ] Movie macro-genre lift quoted correctly
- [ ] Music tag lift quoted correctly
- [ ] Restaurant category lift quoted correctly
- [ ] Movie trustworthiness quoted correctly
- [ ] Music trustworthiness quoted correctly
- [ ] Restaurant trustworthiness quoted correctly

### Cross-domain

- [ ] Matched population sizes reported
- [ ] Method A cross-domain neighbour share reported
- [ ] Method B cross-domain neighbour share reported
- [ ] Relative increase calculated correctly
- [ ] General Semantic vs Feel distinction explained without saying one is universally better

### Feel representation

- [ ] Strong/high dimension correlations acknowledged if relevant
- [ ] Domain effects acknowledged
- [ ] Dimension extremes used as qualitative sanity checks
- [ ] Feel dimensions described as engineered computational descriptors

### Clustering / frontend

- [ ] Giant Feel clusters reported honestly
- [ ] Cluster names described as post-hoc interpretation only
- [ ] 2D nearest-neighbour functionality distinguished from original high-dimensional similarity
- [ ] Limitations of subjective evaluation explicitly discussed

In [ ]:
summary_rows = []

for key, df in tables.items():
    summary_rows.append({
        "table": key,
        "rows": len(df),
        "columns": len(df.columns),
        "numeric_columns": len(df.select_dtypes(include=np.number).columns),
    })

result_inventory = (
    pd.DataFrame(summary_rows)
    .sort_values("table")
    .reset_index(drop=True)
)

display(result_inventory)

---

## End of evaluation summary

At this point the evaluation stage should be considered **frozen**.

If a number in the report needs checking, return to the corresponding CSV/table in this notebook rather than rerunning the modelling pipeline unless an actual error is discovered.